# SNE and phylogenetic embedding overview

Generate the four retained t-SNE coordinate tables used by the downstream R plots.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.manifold import TSNE

repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data/social_niche_embedding_100.txt").exists()
)
analysis_dir = repo_root / "analysis/sne_construction"
output_dir = analysis_dir / "embedding_overview/results/tables"
traits_file = repo_root / "analysis/traits/data/traits_precalculated.txt"
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
def read_embedding(path):
    table = pd.read_csv(path, sep=r'\s+', header=None, index_col=0)
    return table.drop(index='<unk>', errors='ignore')

sne = read_embedding(repo_root / 'data/social_niche_embedding_100.txt')
phylo = read_embedding(repo_root / 'data/phylo_embed_PCA_100.txt')
feature_ids = sne.index.intersection(phylo.index)
sne = sne.loc[feature_ids]
phylo = phylo.loc[feature_ids]

## Full embedding projections


In [ ]:
def project(embedding):
    coordinates = TSNE(n_components=2, metric='cosine', random_state=42).fit_transform(embedding)
    return pd.DataFrame(coordinates, index=embedding.index, columns=['t-SNE1', 't-SNE2'])

project(sne).to_csv(output_dir / 't_sne_co.csv')
project(phylo).to_csv(output_dir / 't_sne_phylo.csv')

## Trait-matched projections


In [ ]:
traits = pd.read_csv(traits_file, sep="	", index_col=0)
trait_ids = feature_ids.intersection(traits.index)
project(sne.loc[trait_ids]).to_csv(output_dir / "t_sne_co_bugbase.csv")
project(phylo.loc[trait_ids]).to_csv(output_dir / "t_sne_phy_bugbase.csv")


## Retained outputs


In [5]:
outputs = [
    output_dir / "t_sne_co.csv",
    output_dir / "t_sne_phylo.csv",
    output_dir / "t_sne_co_bugbase.csv",
    output_dir / "t_sne_phy_bugbase.csv",
]
pd.DataFrame(
    {"Output": [path.name for path in outputs], "Rows": [len(pd.read_csv(path, index_col=0)) for path in outputs]}
)


,Output,Rows
0,t_sne_co.csv,14093
1,t_sne_phylo.csv,14093
2,t_sne_co_bugbase.csv,5832
3,t_sne_phy_bugbase.csv,5832
